In [ ]:
!nvidia-smi

In [ ]:
import pickle
d = "datasets/pre_processed/BallSqueezingHD_modified/sub-170/sub-170_task-BallSqueezing_run-2_nirs.pkl"
with open(d, 'rb') as handle:
    data = pickle.load(handle)
print(data.keys())

data['conc_pcr'].shape, data['delta_conc'].shape, data['rec_stim'].shape, len(data['sensitive_parcels'])

In [ ]:
import os
import numpy as np
import xarray as xr
import pickle
import glob
import warnings
from pathlib import Path, PureWindowsPath
import re

import cedalion
import cedalion.sigproc.motion as motion_correct
import cedalion.sigproc.quality as quality
import cedalion.sigproc.physio as physio
import cedalion.dot as dot
import cedalion.nirs as nirs
import cedalion.vis.anatomy
from cedalion.io.forward_model import load_Adot

from cedalion import units
import pandas as pd

warnings.filterwarnings("ignore")

In [ ]:
def get_bad_ch_mask(int_data, ch_preproc) -> list:
    # Saturated and Dark Channels

    dark_sat_thresh = [1e-3, 0.84]
    amp_threshs_sat = [0., dark_sat_thresh[1]]
    amp_threshs_low = [dark_sat_thresh[0], 1]
    _, amp_mask_sat = quality.mean_amp(int_data, amp_threshs_sat)
    _, amp_mask_low = quality.mean_amp(int_data, amp_threshs_low)
    _, snr_mask = quality.snr(int_data, 10)
    amp_mask=amp_mask_sat & amp_mask_low

    _, list_bad_ch = quality.prune_ch(int_data, [amp_mask, snr_mask], "all")
   
    return list_bad_ch

# Theekshana
# def get_bad_ch_mask(int_data: xr.DataArray, ch_preproc: dict) -> list:
#     # SCI and PSP Mask    
#     sci, sci_mask = quality.sci(int_data, ch_preproc['window_len'], ch_preproc['sci_thresh'])
#     psp, psp_mask = quality.psp(int_data, ch_preproc['window_len'], ch_preproc['psp_thresh'])

#     sci_psp_mask=sci_mask & psp_mask
#     perc_time_clean = sci_psp_mask.sum(dim="time") / len(sci.time)

#     scipsp_bad_ch=[]
#     for ch in perc_time_clean.channel.values:
#         if perc_time_clean.sel(channel=ch).values < ch_preproc['perc_time_clean']: # dont make the mistake of using the inv condition >:| 
#             scipsp_bad_ch.append(ch)

#     sum_bad_ch = scipsp_bad_ch
#     list_bad_ch = sorted(list(set(sum_bad_ch)))  # remove duplicates

#     print("Flagged Channels : ",len(list_bad_ch), '/', len(int_data.channel))
#     print("Percentage: ", int(len(list_bad_ch) / len(int_data.channel) * 100), '%')
    
#     return list_bad_ch

In [ ]:
def standardize_trial_types(DATASET_NAME: str, file: str, stim: pd.DataFrame, rec):
    
    if DATASET_NAME == "FreshMotor":
        # map trial types to left or right depending on the name of the file
        m = re.search(r'(?i)(left|right)', file)

        # rename from MOTOR to left/right
        rec.stim.trial_type = m.group(1).lower()
        # rec.stim = stim
    
    elif DATASET_NAME == "BallSqueezingHD":
        mapping = {
            "Right": "right", # BallSqueezingHD
            "Left": "left",   # BallSqueezingHD
        }
        rec.stim["trial_type"] = rec.stim["trial_type"].replace(mapping)
        
    elif DATASET_NAME == "BS_Laura":
        rec.stim = stim  
        
    elif DATASET_NAME == "Electrical_Thermal":
        mapping = {
            "1": "WordCongruent" ,
            "2": "WordIncongruent",
        }
        rec.stim["trial_type"] = rec.stim["trial_type"].replace(mapping)
        
    elif DATASET_NAME in ["vfc_hd", "Anderson_sparse"]:
        mapping = {
            "1": "WordCongruent" ,
            "2": "WordIncongruent",
        }
        
        rec.stim["trial_type"] = rec.stim["trial_type"].replace(mapping)
    
    rec.stim.sort_values(by="onset", ignore_index=True, inplace=True)

    # attach/update stim info to rec
    # rec.stim = stim

    return stim, rec

# def standardize_trial_types(DATASET_NAME: str, file: str, rec):
    
#     if DATASET_NAME == "FreshMotor":
#         # map trial types to left or right depending on the name of the file
#         m = re.search(r'(?i)(left|right)', file)

#         # rename from MOTOR to left/right
#         rec.stim.trial_type = m.group(1).lower()
    
#     else:
#         mapping = {
#             "Right": "right", # BallSqueezingHD
#             "Left": "left",   # BallSqueezingHD
#             "ElectricalVAS7": "right", # TODO: Electrical_Thermal
#             "ElectricalVAS3": "left",  # TODO: Electrical_Thermal
#         }
#         rec.stim["trial_type"] = rec.stim["trial_type"].replace(mapping)

#     return rec


In [ ]:
 # "full", "subset_2" for spatial_sampling, "motor" for motor_sampling

# subset_type = ""


In [ ]:
mixed_sparse = True
if mixed_sparse:
    n_chs = 50
    subset_type = f"motor_{n_chs}chs" # for sparsified version of Laura
    sparsified_data_path = f'datasets/pre_processed/BS_Laura/channel_subset_BS_Laura_k=2_c3c4k=51_dist4.5_{n_chs}chs.npy'
else:
    subset_type = "full"

base_path = "/home/orabe/fNIRS_sparseToDense/"

# Available datasets:
# DATASET_NAME = "BallSqueezingHD_modified"
# DATASET_NAME = "FreshMotor"
DATASET_NAME = "BS_Laura"
# DATASET_NAME = "ElectricalThermal"
# DATASET_NAME = "vfc_hd"
# DATASET_NAME = "Anderson_sparse"

raw_path = Path(f'datasets/raw/{DATASET_NAME}')
pre_processed_path = Path(f'datasets/pre_processed/{DATASET_NAME}/{subset_type}')

pre_processed_path.mkdir(parents=True, exist_ok=True)
pre_processed_path

In [ ]:
if DATASET_NAME == "BallSqueezingHD_modified":
    raw_dir = f"{raw_path}/sub-*/nirs/sub-*.snirf"

elif DATASET_NAME == "BS_Laura":
    raw_dir = f"{raw_path}/sub-*/nirs/sub-*.snirf"
    
elif DATASET_NAME == "Electrical_Thermal":
    raw_dir = f"{raw_path}/sub-*/ses-*/nirs/sub-*_ses-*_task-Electrical*_nirs.snirf"
    # TODO: exclude subjects without txt files for landmarks coords
    
elif DATASET_NAME == "FreshMotor":
    duration = "*" # * to include both 2s and 3s
    raw_dir = f"{raw_path}/sub-*/ses-*{duration}/nirs/sub-*_ses-*{duration}_task-FRESHMOTOR_nirs.snirf"
elif DATASET_NAME == "vfc_hd":
    # "datasets/raw/vfc_hd/sub-01/nirs/sub-01_ses-02_task-WordStroop_run-01_nirs.snirf"
    raw_dir = f"{raw_path}/sub-*/nirs/sub-*_ses-*_task-WordStroop_run-*_nirs.snirf"
elif DATASET_NAME == "Anderson_sparse":
    # raw_dir = f"{raw_path}/sub-*/ses-*/nirs/sub-*_ses-*_task-WordStroop_run-*_nirs.snirf"
    raw_dir = f"{raw_path}/sub-*/nirs/sub-*_ses-*_task-WordStroop_run-*_nirs.snirf"    

else:
    raise ValueError("Unknown dataset name")

files = glob.glob(raw_dir)

# TODO: to be confirmed
# remove non-BS files for Laura's dataset to avoid errors
if DATASET_NAME == "BS_Laura":
    files = [p for p in files if "BS" in os.path.basename(p)]
    # remove files that has this pattern in the name: _acq-4NN_nirs:
    files = [p for p in files if "_acq-4NN_nirs" not in os.path.basename(p)]
    
files = sorted(files)
print(f"{len(files)} files found.")

In [ ]:
files[0]

In [ ]:
# spatial_sampling is the approach we use to subsample channels based on predifined spatial locations. See src/subset/optode_subsets.ipynb
spatial_sampling = False
if spatial_sampling:
    # This is a temoraly code to load subset channels. For FreshMotor we don't have subsets yet as we need all channels. Thus we do construct subset_channels manually and make it equal to all channels.
    if DATASET_NAME == "BallSqueezingHD_modified":
        with open(f"results/subset/{DATASET_NAME}/subsets_data.pkl", "rb") as f:
            subsets_data = pickle.load(f)
        subset_channels = subsets_data[subset_type]["all"]
        
    elif DATASET_NAME == "FreshMotor":
        # make subset_channels equal to all channels
        filename = files[0] # select one
        rec = cedalion.io.read_snirf(filename)[0]  # read snirf files
        all_channels = rec['amp']['channel'].values.tolist()
        subset_channels = all_channels

# motor_sampling is the approach we use to subsample channels based on motor area locations. See src/subset/sparsify_chs_from_sens.ipynb
motor_sampling = True
if motor_sampling:
    if DATASET_NAME == "BS_Laura":
        subset_channels = np.load(sparsified_data_path, allow_pickle=True)
        
no_subsampling = False
if no_subsampling:
     # make subset_channels equal to all channels
    filename = files[0] # select one
    print(filename)
    rec = cedalion.io.read_snirf(filename)[0]  # read snirf files
    all_channels = rec['amp']['channel'].values.tolist()
    subset_channels = all_channels
    
print(len(subset_channels))

In [ ]:
filename = files[0] # select one
rec = cedalion.io.read_snirf(filename)[0]  # read snirf files

# subset the data
rec['amp'] = rec['amp'].sel(channel=subset_channels)

# subset measurement list
meas_list = rec._measurement_lists["amp"]
meas_list = meas_list[meas_list["channel"].isin(subset_channels)].reset_index(drop=True)

# both should then be equal
print(f'Number of channels in rec["amp"]: {len(set(rec["amp"].channel.values))}')
print(f'Number of channels in meas_list: {len(set(meas_list.channel.values))}')

head_icbm152 = dot.get_standard_headmodel('icbm152')  


if DATASET_NAME == "BS_Laura":
    # this is required for BU Data (Laura's)
    T = np.array([
        [-9.57882733e-01, -7.20806358e-03,  6.20193531e-03, 2.21208571e+02],
        [-2.02271710e-02,  6.03819925e-02,  9.94046165e-01, -2.03010603e+01],
        [-8.79481533e-03, -1.02761992e+00,  6.59199998e-02, 2.87749135e+02],
        [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00, 1.00000000e+00]])
    
    ninja_aligned = rec.geo3d.points.apply_transform(T)
    geo3d_snapped_ijk = head_icbm152.align_and_snap_to_scalp(ninja_aligned)
else:
    geo3d_snapped_ijk = head_icbm152.align_and_snap_to_scalp(rec.geo3d)
    

fwm = cedalion.dot.forward_model.ForwardModel(
    head_icbm152, 
    geo3d_snapped_ijk,
    meas_list
)

fluence_fname = os.path.join(pre_processed_path, "fluence_" + DATASET_NAME + ".h5")
sensitivity_fname = os.path.join(pre_processed_path, "sensitivity_" + DATASET_NAME + ".h5")

# compute fluence and sensitivity only once
fwm.compute_fluence_mcx(fluence_fname)
fwm.compute_sensitivity(fluence_fname, sensitivity_fname)

Adot = load_Adot(sensitivity_fname)

recon = dot.ImageRecon(
    Adot,
    recon_mode="mua2conc",
    brain_only=True,
    alpha_meas=10,
    alpha_spatial=10e-3,
    apply_c_meas=True,
    spatial_basis_functions=None,
)

In [ ]:
# import cedalion.vis.blocks as vbx
# import pyvista as pv
# import cedalion.dataclasses as cdc
# # keep only points that are not of type "landmark", i.e. source and detector points
# geo3d_snapped_ijk = geo3d_snapped_ijk[geo3d_snapped_ijk.type != cdc.PointType.LANDMARK]

# # now we plot the head same as before...
# plt = pv.Plotter()
# vbx.plot_surface(plt, head_icbm152.brain, color="#d3a6a1")
# vbx.plot_surface(plt, head_icbm152.scalp, opacity=.1)
# # but use the plot_labeled_points() function to add the snapped geo3d.
# # The flag "show_labels" can be used to show the source, detector, and landmark names
# vbx.plot_labeled_points(plt, geo3d_snapped_ijk, show_labels=True)
# plt.show()
# # save image
# plt.screenshot("head_with_snapped_points.png")

In [ ]:
# import cedalion.vis.blocks as vbx
# from cedalion.io.forward_model import FluenceFile

# # pull fluence values from the corresponding source and detector pair

# with FluenceFile(fluence_fname) as fluence_file:
#     f = fluence_file.get_fluence("S12", 760) * fluence_file.get_fluence("D19", 760)

# f = np.log10(np.clip(f, min=f[f > 0].min()))
# vf = pv.wrap(f)

# plt = pv.Plotter()

# plt.add_volume(
#     vf,
#     log_scale=False,
#     cmap="plasma_r",
#     clim=(-10, -0),
#     scalar_bar_args={
#         "title": r"$log_{10}("
#         r"F(\vec{x}_{src},\vec{x}) * F(\vec{x}, \vec{x}_{det})"
#         ")$"
#     },
# )
# vbx.plot_surface(plt, head_icbm152.brain, color="w")
# vbx.plot_labeled_points(plt, geo3d_snapped_ijk, show_labels=False)
# vbx.camera_at_cog(plt, head_icbm152.brain, rpos=[300, 150, 150])
# plt.show()
# plt.screenshot("fluence_visualization.png")

In [ ]:

# import cedalion.vis.anatomy.sensitivity_matrix as sensitivity_matrix

# plotter = sensitivity_matrix.Main(
#     sensitivity=Adot,
#     brain_surface=head_icbm152.brain,
#     head_surface=head_icbm152.scalp,
#     labeled_points=geo3d_snapped_ijk,
# )
# plotter.plot(high_th=0, low_th=-3)
# plotter.plt.show()
# # save the figure
# plotter.plt.screenshot("sensitivity_visualization.png")

In [ ]:
pre_processed_path

In [ ]:
print(len(meas_list['channel']))
print(rec["amp"].channel.size)
print(recon._F.shape)

In [ ]:
stim = cedalion.io.read_events_from_tsv(files[0].replace('nirs.snirf', 'events.tsv'))

stim, rec = standardize_trial_types(DATASET_NAME, files[0], stim, rec)
rec.stim.head()

In [ ]:
print(np.diff(np.sort(rec.stim.onset.values)))
print(np.min(np.diff(np.sort(rec.stim.onset.values))))

In [ ]:
from cedalion.io import read_events_from_tsv

subject_to_rec = {}
skipped_subjects = []

file_sens_drop_parcels_dict = {}
for i, f in enumerate (files):
    print(f"--- Processing file: {i+1}/{len(files)} ---")

    if DATASET_NAME == "vfc_hd":
        subj = PureWindowsPath(f).parts[-3]
        if subj == "sub-13":
            continue # skip bad subject

    records = cedalion.io.read_snirf(f)
    rec = records[0]

    # select subset channels
    rec['amp'] = rec['amp'].sel(channel=subset_channels)
    
    # load event files
    stim = read_events_from_tsv(f.replace('nirs.snirf', 'events.tsv'))
    
    # sort
    rec.stim = rec.stim.sort_values(by="onset")
    # stim = stim.sort_values(by="onset")
    
    stim, rec = standardize_trial_types(DATASET_NAME, f, stim, rec)

    rec['rep_amp'] = quality.repair_amp(rec['amp'], median_len=3, method='linear')  # Repair Amp
    rec['od_amp'], baseline= nirs.cw.int2od(rec['rep_amp'],return_baseline=True)

    # motion correct [TDDR + WAVELET]
    rec["od_tddr"] = motion_correct.tddr(rec["od_amp"])
    rec["od_tddr_wavel"] = motion_correct.wavelet(rec["od_tddr"])

    # -----------------------------------highpass filter--------------------------------
    rec['od_hpfilt'] = rec['od_tddr_wavel'].cd.freq_filter(fmin=0.008,fmax=0,butter_order=4)
    # ----------------------------------------------------------------------------------

    # clean amplitude data
    rec['amp_clean'] = cedalion.nirs.cw.od2int(rec['od_hpfilt'], baseline)

    # get bad channel mask
    # list_bad_ch = get_bad_ch_mask(rec["amp_clean"]) # this has custom paramerers!? 
    
    ch_preproc = {
        'sci_thresh' : 0.5, # 0.75, # 0.6, #0.5 orignal
        'psp_thresh' : 0.1, # 0.03, #0.08, #0.1 orignal
        'window_len' : 5*units.s,           # 5 seconds
        'dark_sat_thresh' : [1e-3, 0.84],
        'perc_time_clean' : 0.5             # 50 %   
    }  
    list_bad_ch = get_bad_ch_mask(rec["amp_clean"], ch_preproc) # theekshana
    
    print('the list of bad channels: ', len(list_bad_ch))

    # channel variance
    od_var_vec = quality.measurement_variance(rec["od_hpfilt"], list_bad_channels=list_bad_ch, bad_rel_var=1e6,calc_covariance=False)

    # ----------------------------------------------------------------------------------
    dpf = xr.DataArray(
        [6, 6],
        dims="wavelength",
        coords={"wavelength": rec["amp"].wavelength},
    )
    rec['conc'] = cedalion.nirs.cw.od2conc(rec['od_hpfilt'], rec.geo3d, dpf, spectrum="prahl")

    # conc_pr vs conc 
    try:
        chromo_var = quality.measurement_variance(rec['conc'], list_bad_channels = list_bad_ch, bad_rel_var = 1e6, calc_covariance = False)
    except Exception as e:
        print(f"Skipping file {f} due to error in chromo_var calculation: {e}")
        skipped_subjects.append(f)
        continue
    rec['conc_pcr'], gb_comp_rem = physio.global_component_subtract(rec['conc'],ts_weights=1/chromo_var,k=0,spatial_dim='channel',spectral_dim='chromo')

    rec['od_pcr1'] = cedalion.nirs.cw.conc2od(rec['conc_pcr'], rec.geo3d, dpf, spectrum="prahl")#     delta_conc = chunked_eff_xr_matmult(od_stacked, B, contract_dim="flat_channel", sample_dim="time", chunksize=300)
    
    c_meas = quality.measurement_variance(
        rec['od_hpfilt'],
        list_bad_channels=list_bad_ch,
        bad_rel_var=1e6,
        calc_covariance=False
    )

    delta_conc = recon.reconstruct(rec['od_pcr1'], c_meas) 
    delta_conc.time.attrs["units"] = units.s

    dC_brain = delta_conc.cd.freq_filter(fmin=0.01, fmax=0.5, butter_order=4)
    dC_brain = dC_brain.sel(time=slice(rec.stim.onset.values[0]-3 , rec.stim.onset.values[-1]+13))
    dC_brain = dC_brain.where(dC_brain.is_brain == True)
    # alternatively use 1/conc_var to weight vertex sensitivity and then normalize by sum of weights
    dC_brain = dC_brain.pint.quantify().pint.to("uM").pint.dequantify()

    hbr = dC_brain.sel(chromo='HbR').groupby('parcel').mean()
    hbo = dC_brain.sel(chromo='HbO').groupby('parcel').mean()
    signal_raw = xr.concat([hbo, hbr], dim='chromo')

    # revised matrix
    signal_raw = signal_raw.sel(parcel=signal_raw.parcel != 'Background+FreeSurfer_Defined_Medial_Wall_LH')
    signal_raw = signal_raw.sel(parcel=signal_raw.parcel != 'Background+FreeSurfer_Defined_Medial_Wall_RH')
    
    delta_conc, global_comp = physio.global_component_subtract(
        signal_raw, 
        ts_weights=None, k=0, 
        spatial_dim='parcel',
        spectral_dim= 'chromo')

    delta_conc = delta_conc / np.abs(delta_conc).max()
    delta_conc = delta_conc.fillna(0)
    delta_conc = delta_conc.transpose("time", "parcel", "chromo")

    parcel_dOD, parcel_mask = fwm.parcel_sensitivity(
        Adot,
        [], # list_bad_ch,
        dOD_thresh = 0.001,       
        minCh=1,
        dHbO=10,
        dHbR=-3
    )
    sensitive_parcels = parcel_mask.where(parcel_mask, drop=True)["parcel"].values.tolist()
    dropped_parcels = parcel_mask.where(~parcel_mask, drop=True)["parcel"].values.tolist()
    file_sens_drop_parcels_dict[f] = {
        "sensitive_parcels": sensitive_parcels,
        "dropped_parcels": dropped_parcels
    }
    print(f"Number of sensitive parcels: {len(sensitive_parcels)}")
    print(f"Number of dropped parcels: {len(dropped_parcels)}")
    
    total_parcels = len(sensitive_parcels) + len(dropped_parcels)
    dropped_ratio = (len(dropped_parcels) / total_parcels) if total_parcels else 1.0
    
    print(f"Dropped parcels ratio: {dropped_ratio:.2f}")
    if dropped_ratio > 0.99:
        print(f"Skipping {f}: dropped {len(dropped_parcels)} / {total_parcels}, remaining {len(sensitive_parcels)}")
        skipped_subjects.append(f)
        continue
        
    data = {
        'conc_pcr': rec['conc_pcr'],
        'delta_conc': delta_conc,
        'rec_stim': rec.stim,
        'sensitive_parcels': sensitive_parcels,
    }
    
    # save as pickle (all parcels!!!)
    path = PureWindowsPath(f)
    subject_dir = path.parts[-3]
    filename = path.stem

    if DATASET_NAME == "FreshMotor":
        subject_dir = path.parts[-4]
        session_label = path.parts[-3]
        task_fragment = next(
            (part for part in filename.split('_') if part.startswith('task-')),
            f"task-{DATASET_NAME.replace('_', '').upper()}",
        )
        run_fragment = session_label.replace('ses-', 'run-')
        filename = f'{subject_dir}_{task_fragment}_{run_fragment}_nirs'

    if subject_dir not in subject_to_rec:
        subject_to_rec[subject_dir] = []

    all_parcels_dir = pre_processed_path / subject_dir
    all_parcels_dir.mkdir(parents=True, exist_ok=True)

    file_name_to_save = all_parcels_dir / f'{filename}.pkl'

    with open(file_name_to_save, 'wb') as handle:
        pickle.dump(data, handle, protocol=pickle.HIGHEST_PROTOCOL)

if skipped_subjects:
    print(f"Skipped {len(skipped_subjects)} file(s)")

print("\nProcessing complete.")


In [ ]:
f

In [ ]:
with open(pre_processed_path / "file_sens_drop_parcels_dict.pkl", 'wb') as handle:
    pickle.dump(file_sens_drop_parcels_dict, handle, protocol=pickle.HIGHEST_PROTOCOL)  

In [ ]:
# sens_parcel_template_path = 'datasets/parcel_templates/parcel_template_vfc_hd.pkl'
# sens_parcel_template_path = 'datasets/parcel_templates/parcel_template_Anderson_sparse.pkl'
sens_parcel_template_path = 'datasets/parcel_templates/parcel_template_BallSqueezingHD_modified.pkl'

with open(sens_parcel_template_path, 'rb') as handle:
    template_sens_parcel_list = pickle.load(handle)
    
print(len(template_sens_parcel_list))

# for each subject, check how many of the sensitive parcels from the dictionary are contained in the template parcel list
for f, sens_drop_info in file_sens_drop_parcels_dict.items():
    sensitive_parcels = sens_drop_info['sensitive_parcels']
    
    num_sensitive_in_template = sum(1 for parcel in sensitive_parcels if parcel in template_sens_parcel_list)
    num_dropped_in_template = sum(1 for parcel in dropped_parcels if parcel in template_sens_parcel_list)
    
    # print(f"File: {f}")
    print(f"  Sensitive parcels in template: {num_sensitive_in_template} / {len(template_sens_parcel_list)}")


In [ ]:
1

In [ ]:
import matplotlib.pyplot as plt
# Plot as barplot the number of sensitive parcels in the template for each subject
files = list(file_sens_drop_parcels_dict.keys())
num_sensitive_in_template_list = []
for f in files:
    sensitive_parcels = file_sens_drop_parcels_dict[f]['sensitive_parcels']
    num_sensitive_in_template = len(set(sensitive_parcels) & set(template_sens_parcel_list))
    num_sensitive_in_template_list.append(num_sensitive_in_template)
plt.figure(figsize=(10, 6))
plt.bar(range(len(files)), num_sensitive_in_template_list, color='blue')
# print the number of sensitive parcels in the template for each subject on top of the bars
for i, num in enumerate(num_sensitive_in_template_list):    
    plt.text(i, num + 0.5, str(num), ha='center', va='bottom')
# plt.plot(range(len(files)), num_sensitive_in_template_list, color='red')
plt.xticks(range(len(files)), [f.split('/')[3].replace('.snirf', '') for f in files], rotation=45)
plt.xlabel('Subjects')
plt.ylabel('Number of Sensitive Parcels in Template')
plt.ylim(0, len(template_sens_parcel_list) + 10)
plt.axhline(y=len(template_sens_parcel_list), color='gray', linestyle='dashed')
# plt.legend()
# plt.title('Sensitive Parcels in Template for Each Subject')
plt.tight_layout()
plt.show()

In [ ]:
len(files), len(skipped_subjects)

In [ ]:
rec['amp_clean']

In [ ]:
rec.stim 

In [ ]:
with open("datasets/pre_processed/vfc_hd/sub-01/sub-01_ses-02_task-WordStroop_run-01_nirs.pkl", 'rb') as handle:
    data = pickle.load(handle)
data['rec_stim']

In [ ]:
# load data for visualization
if DATASET_NAME == "BallSqueezingHD_modified":
    subject_name = 'sub-185'
    file_name = f'{subject_name}_task-BallSqueezing_run-3_nirs.pkl'
    file_to_plot = pre_processed_path / subject_name / file_name
    
elif DATASET_NAME == "FreshMotor":
    subject_name = 'sub-01'
    file_name = f'{subject_name}_task-FRESHMOTOR_run-left2s_nirs.pkl'
    file_to_plot = pre_processed_path / subject_name / file_name

elif DATASET_NAME == "BS_Laura":
    subject_name = 'sub-538'
    file_name = f'{subject_name}_task-BS_run-01_nirs.pkl'
    file_to_plot = pre_processed_path / subject_name / file_name
elif DATASET_NAME == "vfc_hd":
    subject_name = 'sub-12'
    file_name = f'{subject_name}_ses-01_task-WordStroop_run-01_nirs.pkl'
    file_to_plot = pre_processed_path / subject_name / file_name
elif DATASET_NAME == "Anderson_sparse":
    subject_name = 'sub-1'
    file_name = f'{subject_name}_ses-1_task-WordStroop_run-1_nirs.pkl'
    file_to_plot = pre_processed_path / subject_name / file_name    
    
# load data
with open(file_to_plot, 'rb') as handle:
    data = pickle.load(handle)

In [ ]:
data['delta_conc']

In [ ]:
data.keys()

# Calculate block averages in optical density



In [ ]:
data['conc_pcr']

In [ ]:
rec['od_pcr1']

Blockaverage od_pcr1

In [ ]:
# import matplotlib.pyplot as p

# # segment data into epochs
# if DATASET_NAME in ["BallSqueezingHD_modified", "BS_Laura"]:
#     epochs = rec['od_pcr1'].cd.to_epochs(
#         rec.stim,  # stimulus dataframe
#         ["left", "right"],  # select fingertapping events, discard others
#         before=2 * units.s,  # seconds before stimulus
#         after=10 * units.s,  # seconds after stimulus
#     )
# elif DATASET_NAME == "FreshMotor":
#     epochs = rec['od_pcr1'].cd.to_epochs(
#         rec.stim,  # stimulus dataframe
#         ["right"],  # select left/right events
#         before=2 * units.s,  # seconds before stimulus
#         after=10 * units.s,  # seconds after stimulus
#     )
# elif DATASET_NAME in ["vfc_hd", "Anderson_sparse"]:
#     epochs = rec['od_pcr1'].cd.to_epochs(
#         rec.stim,  # stimulus dataframe
#         ["WordCongruent", "WordIncongruent"],
#         before=2 * units.s,  # seconds before stimulus
#         after=10 * units.s,  # seconds after stimulus
#     )
    
# # calculate baseline
# baseline = epochs.sel(reltime=(epochs.reltime < 0)).mean("reltime")

# # subtract baseline
# epochs_blcorrected = epochs - baseline

# # group trials by trial_type. For each group individually average the epoch dimension
# blockaverage_od_pcr1 = epochs_blcorrected.groupby("trial_type").mean("epoch")

In [ ]:
# from IPython.display import Image
# def display_image(fname : str):
#     display(Image(data=open(fname,'rb').read(), format='png'))

In [ ]:
# results_path_prefix = f'results/{subset_type}/{DATASET_NAME}/{subject_name}'
# os.makedirs(results_path_prefix, exist_ok=True)

In [ ]:
# # Plot block averages. Please ignore errors if the plot is too small in the HD case

# filename = f"results/{subset_type}/{DATASET_NAME}/blockaverage_channel_space_{subset_type}.png"

# noPlts2 = int(np.ceil(np.sqrt(len(blockaverage_od_pcr1.channel))))
# f,ax = p.subplots(noPlts2,noPlts2, figsize=(12,10))
# ax = ax.flatten()
# for i_ch, ch in enumerate(blockaverage_od_pcr1.channel):
#     for ls, trial_type in zip(["-", "--"], blockaverage_od_pcr1.trial_type):
#         ax[i_ch].plot(blockaverage_od_pcr1.reltime, blockaverage_od_pcr1.sel(wavelength=760, trial_type=trial_type, channel=ch), "r", lw=2, ls=ls)
#         ax[i_ch].plot(blockaverage_od_pcr1.reltime, blockaverage_od_pcr1.sel(wavelength=850, trial_type=trial_type, channel=ch), "b", lw=2, ls=ls)

#     ax[i_ch].grid(1)
#     ax[i_ch].set_title(ch.values)
#     ax[i_ch].set_ylim(-.02, .02)
#     ax[i_ch].set_axis_off()
#     ax[i_ch].axhline(0, c="k")
#     ax[i_ch].axvline(0, c="k")

# # p.suptitle("760nm: r | 850nm: b | left: - | right: --")
# p.suptitle("HbO: r | HbR: b | left: - | right: --")

# p.tight_layout()
# p.savefig(filename)

In [ ]:
def match_landmark_labels(rec):
    subject_nasion_mask = rec.geo3d['label'].data == 'NASION'

    new_labels = rec.geo3d['label'].data.copy()
    new_labels[subject_nasion_mask] = 'Nz'

    # Create new geo3d with updated labels
    rec.geo3d = rec.geo3d.assign_coords(label=new_labels)
    # print(rec.geo3d['label'].data)

    return rec

In [ ]:
rec = match_landmark_labels(rec)

In [ ]:
# # Viz reconstruction on Channel Space
# import cedalion.vis.anatomy
# filename_scalp = f"results/{subset_type}/{DATASET_NAME}/scalp_plot_ts_{subset_type}.png"

# # data_ts = blockaverage_od_pcr1.sel(wavelength=850, trial_type="right")
# data_ts = blockaverage_od_pcr1.sel(wavelength=850, trial_type="WordCongruent")
# # scalp_plot_gif expects the time dimension to be named 'time'
# data_ts = data_ts.rename({"reltime": "time"})

# # call plot function 
# cedalion.vis.anatomy.scalp_plot_gif(
#     data_ts,
#     rec.geo3d,
#     filename=filename_scalp,
#     time_range=(-5, 30, 0.5) * units.s,
#     scl=(-0.01, 0.01),
#     fps=6,
#     optode_size=6,
#     optode_labels=True,
#     str_title="OD 850 nm",
# )
# display_image(f"{filename_scalp}.gif")

Blockaverage_delta_conc

In [ ]:
# if DATASET_NAME == "BallSqueezingHD_modified" or DATASET_NAME == "BS_Laura":
#     epochs = data['delta_conc'].cd.to_epochs(
#         rec.stim,  # stimulus dataframe
#         ["left", "right"],  # select fingertapping events, discard others
#         before=2 * units.s,  # seconds before stimulus
#         after=10 * units.s,  # seconds after stimulus
#     )
# elif DATASET_NAME == "FreshMotor":
#     # segment data into epochs
#     epochs = data['delta_conc'].cd.to_epochs(
#         rec.stim,  # stimulus dataframe
#         ["right"],  # select fingertapping events, discard others
#         before=2 * units.s,  # seconds before stimulus
#         after=10 * units.s,  # seconds after stimulus
#     )
    
# elif DATASET_NAME in ["vfc_hd", "Anderson_sparse"]:
#     # segment data into epochs
#     epochs = data['delta_conc'].cd.to_epochs(
#         rec.stim,  # stimulus dataframe
#         ["WordCongruent", "WordIncongruent"],  # select fingertapping events, discard others
#         before=2 * units.s,  # seconds before stimulus
#         after=10 * units.s,  # seconds after stimulus
#     )    

# # calculate baseline
# baseline = epochs.sel(reltime=(epochs.reltime < 0)).mean("reltime")

# # subtract baseline
# epochs_blcorrected = epochs - baseline

# # group trials by trial_type. For each group individually average the epoch dimension
# blockaverage_delta_conc = epochs_blcorrected.groupby("trial_type").mean("epoch")

In [ ]:
# vertex_parcels = head_icbm152.brain.vertex_coords['parcel']
# vertex_parcels = np.array(vertex_parcels)

# parcel_index = blockaverage_delta_conc.get_index("parcel")  # pandas index
# vertex_parcel_idx = parcel_index.get_indexer(vertex_parcels)

# parcel_data = blockaverage_delta_conc.values  # shape (2, 107, 601, 2)

# # Broadcast using integer indexing on axis=2 (parcel axis)
# vertex_activity = parcel_data[:, :, vertex_parcel_idx, :]

# n_vertices = len(vertex_parcel_idx)

# vertex_da = xr.DataArray(
#     vertex_activity,
#     dims=("trial_type", "reltime", "vertex", "chromo"),
#     coords=dict(
#         trial_type=blockaverage_delta_conc.trial_type,
#         reltime=blockaverage_delta_conc.reltime,
#         chromo=blockaverage_delta_conc.chromo,

#         # vertex index
#         vertex=np.arange(n_vertices),

#         # parcel label of each vertex
#         parcel_of_vertex=("vertex", vertex_parcels),

#         # NEW: is_brain flag
#         is_brain=("vertex", np.ones(n_vertices, dtype=bool))
#     )
# )

# vertex_da.shape

In [ ]:
# filename_multiview = f'{results_path_prefix}/image_recon_multiview_{subset_type}'

# # prepare data
# # X_ts = vertex_da.sel(trial_type="right").rename({"reltime": "time"})
# X_ts = vertex_da.sel(trial_type="WordCongruent").rename({"reltime": "time"})
# X_ts = X_ts.transpose("vertex", "chromo", "time")

# scl = np.percentile(np.abs(X_ts.sel(chromo='HbO')).pint.dequantify(), 99)
# clim = (-scl,scl)

# cedalion.vis.anatomy.image_recon_multi_view(
#     X_ts,  # time series data; can be 2D (static) or 3D (dynamic)
#     head_icbm152,
#     cmap='seismic',
#     clim=clim,
#     view_type='hbo_brain',
#     title_str='HbO / µM',
#     filename=filename_multiview,
#     SAVE=True,
#     time_range=(-2,10,0.5)*units.s,
#     fps=6,
#     geo3d_plot = None, #  geo3d_plot
#     wdw_size = (1024, 768)
# )
# display_image(filename_multiview+'.gif')

# Segmentation

In [ ]:
pre_processed_path

In [ ]:
# load (all) parcel files
preproc_files_path = str(pre_processed_path / 'sub-*' / '*.pkl')
proc_pkl_files = glob.glob(preproc_files_path)

len(proc_pkl_files), proc_pkl_files[:2]

In [ ]:
processed_path = Path(f'datasets/processed/{DATASET_NAME}/{subset_type}')
processed_path.mkdir(parents=True, exist_ok=True)
processed_path

### Create template sensitive parcels

In [ ]:
create_orload_parcel_template = "load"  # "load" or "create"

if create_orload_parcel_template == "create":
    # Load any file as all processed files have the same sensitive parcels
    # f = f"datasets/{subset_type}_pre_processed/BallSqueezingHD_modified/sub-185/sub-185_task-BallSqueezing_run-1_nirs.pkl"
    
    if DATASET_NAME == "BallSqueezingHD_modified":
        f = f"datasets/pre_processed/BallSqueezingHD_modified/sub-185/sub-185_task-BallSqueezing_run-1_nirs.pkl"
    elif DATASET_NAME == "vfc_hd":
        f = f"datasets/pre_processed/vfc_hd/sub-01/sub-01_ses-02_task-WordStroop_run-01_nirs.pkl"
    elif DATASET_NAME == "Anderson_sparse":
        f = f"datasets/pre_processed/Anderson_sparse/sub-1/sub-1_ses-1_task-WordStroop_run-1_nirs.pkl"
        
    with open(f, 'rb') as handle:
        data = pickle.load(handle)
        
    print(len(data['sensitive_parcels']))

    # save the loaded sensitive parcels as template as pkl file
    folder_path = 'datasets/parcel_templates'
    os.makedirs(folder_path, exist_ok=True)
    # sens_parcel_template_path = os.path.join(folder_path, f'parcel_template_{DATASET_NAME}.pkl')
    # sens_parcel_template_path = os.path.join(folder_path, f'{subset_type}_parcel_template_subset_2_BallSqueezingHD_modified.pkl')
    # sens_parcel_template_path = "datasets/parcel_templates/parcel_template_vfc_hd.pkl"
    sens_parcel_template_path = "datasets/parcel_templates/parcel_template_Anderson_sparse.pkl"

    with open(sens_parcel_template_path, 'wb') as handle:
        pickle.dump(data['sensitive_parcels'], handle)
    
    print(f"Parcel template saved to {sens_parcel_template_path}")
        
        
elif create_orload_parcel_template == "load":
    folder_path = 'datasets/parcel_templates'
    
    # sens_parcel_template_path = os.path.join(folder_path, 'subset_2_parcel_template_subset_2_BallSqueezingHD_modified.pkl')
    sens_parcel_template_path = os.path.join(folder_path, 'parcel_template_BallSqueezingHD_modified.pkl')
    # sens_parcel_template_path = os.path.join(folder_path, 'parcel_template_vfc_hd.pkl')
    # sens_parcel_template_path = os.path.join(folder_path, 'parcel_template_Anderson_sparse.pkl')
    
    with open(sens_parcel_template_path, 'rb') as handle:
        template_sens_parcel_list = pickle.load(handle)
        
    print(len(template_sens_parcel_list))
    print(f"Parcel template loaded from {sens_parcel_template_path}")

In [ ]:
# check how many of the sensitive parcels are motor are
motor_labels = [l for l in template_sens_parcel_list if l.startswith("SomMot")]
print(f"{len(motor_labels)} motor labels")

In [ ]:
template_sens_parcel_list

In [ ]:
min_len_sens_parcels = float('inf')
min_len_i = None

for i in range(len(proc_pkl_files)):
    with open(proc_pkl_files[i], 'rb') as handle:
        data_pickle = pickle.load(handle)
        delta_brain = data_pickle['delta_conc']
        sensitive_parcels = data_pickle['sensitive_parcels']
        print(f"File {i}: {len(sensitive_parcels)} sensitive parcels")
        if len(sensitive_parcels) < min_len_sens_parcels:
            min_len_sens_parcels = len(sensitive_parcels)
            min_len_i = i
            
        # also check how many of the sensitive parcels are motor are
        motor_labels = [l for l in sensitive_parcels if l.startswith("SomMot")]
        print(f"File {i}: {len(motor_labels)} motor labels")
        print("-"*50)
            
min_len_i, min_len_sens_parcels

In [ ]:
with open(proc_pkl_files[7], 'rb') as handle:
    data_pickle = pickle.load(handle)
    delta_brain = data_pickle['delta_conc']
    sensitive_parcels = data_pickle['sensitive_parcels']
len(sensitive_parcels)

In [ ]:
from collections import Counter

# Gather all fs_mean values from all files
fs_list = []
for file in proc_pkl_files:
    with open(file, 'rb') as handle:
        data_pickle = pickle.load(handle)
    delta_brain = data_pickle['delta_conc']
    
    dt = np.diff(delta_brain.time.data)
    fs_mean = float(1.0 / np.mean(dt))
    
    fs_list.append(fs_mean)

fs_counts = Counter(fs_list)
len (proc_pkl_files), fs_counts

In [ ]:
with open(file, 'rb') as handle:
    data_pickle = pickle.load(handle)

delta_brain = data_pickle['delta_conc']
sensitive_parcels = data_pickle['sensitive_parcels']
rec_stim = data_pickle['rec_stim']

delta_brain.time.data

In [ ]:
delta_brain.sel(parcel=template_sens_parcel_list).shape

In [ ]:
baseline_duration = 2.5  # in seconds
n_shifts = 9
duration = 10  # in seconds
post_padding = 5  # in seconds
n_timepoints = 87 # 244 # 174 # 87  # fixed length after shifting
# Todo: downsample to 87 Hz 
resample = True

# if DATASET_NAME == "BallSqueezingHD_modified" or DATASET_NAME == "BS_Laura":
if DATASET_NAME in ["BallSqueezingHD_modified",
                    "BS_Laura",
                    "vfc_hd",
                    "Anderson_sparse"]:
    delta_range = (-2.5, 2.5)
elif DATASET_NAME == "FreshMotor":
    delta_range = (-2.0, 0.0)
start_shift = np.linspace(*delta_range, n_shifts)
    
for file in proc_pkl_files:
    with open(file, 'rb') as handle:
        data_pickle = pickle.load(handle)
    
    delta_brain = data_pickle['delta_conc']
    sensitive_parcels = data_pickle['sensitive_parcels']
    rec_stim = data_pickle['rec_stim']

    # [OLD]: Align subject-specific parcels to a common parcel template (zero-pad missing parcels)
    # delta_brain = delta_brain.sel(parcel=sensitive_parcels).reindex(parcel=PARCEL_TEMPLATE, fill_value=0)
    
    # [NEW]: select only parcels in the common template
    delta_brain = delta_brain.sel(parcel=template_sens_parcel_list)
    
    
    # ---- NEW: Resampling ----
    if resample:
        target_fs = 8.7 # 24.4 # 8.98876404494382  # From BSQ-HD
        dt = 1.0 / target_fs
        t0 = float(delta_brain.time.min())
        t1 = float(delta_brain.time.max())
        new_time = np.arange(t0, t1 + 1e-9, dt)
        delta_brain = delta_brain.interp(time=new_time)
    # -------------------------
    
    # Process event segments
    i = 0
    for index, row in rec_stim.iterrows():
        # Binary labeling: pool word conditions into "task" label
        if "word" in row["trial_type"].lower():
            label = "task"
        else:
            label = row["trial_type"].lower()
        # print(f"{row['trial_type']} -> {label}")
        for s in start_shift:
            start_time = row["onset"] + s
            end_time = start_time + duration + post_padding # in seconds
            baseline = delta_brain.sel(
                time=slice(row["onset"] - baseline_duration, row["onset"])
            ).mean("time")
            
            # Then, trimming is easy with `.sel()`:
            x = delta_brain.sel(time=slice(start_time, end_time)) - baseline
            x = x.isel(time=slice(0, n_timepoints))
            
            # new check
            # print(x.sizes["time"], " timepoints")
            if x.sizes["time"] < n_timepoints:
                print(f"Skipping segment for file {os.path.basename(file)} at trial {i} due to insufficient length: {x.sizes['time']} < {n_timepoints}")
                # continue  # skip short segment
            
            x = x.transpose("parcel", "chromo", "time")
            del x.time.attrs['units']

            if not os.path.exists(os.path.dirname(file.replace(str(pre_processed_path), str(processed_path)))):
                os.makedirs(os.path.dirname(file.replace(str(pre_processed_path), str(processed_path))))
            if s == 0:
                x.to_netcdf(file.replace(str(pre_processed_path), str(processed_path)).replace(".pkl", "_" + label + "_"+str(i)+"_test.nc"))
                i += 1
            else:
                x.to_netcdf(file.replace(str(pre_processed_path), str(processed_path)).replace(".pkl", "_" + label + "_"+str(i)+".nc"))
                i += 1

    if False:
        # --- REST-STATE SEGMENT EXTRACTION ---
        # Rest segments: start at event_onset + 15s, end at next_event_onset - 5s
        # Extracted with 0.5s sliding window, 10s duration each
        recording_start = float(delta_brain.time.values[0])
        recording_end = float(delta_brain.time.values[-1])
        segment_length_sec = duration  # 10 seconds
        rest_label = "rest"
        step_sec = 0.25  # sliding window step
        rest_segment_count = 0
        
        # Process rest intervals between consecutive events
        for idx in range(len(rec_stim)):
            current_onset = rec_stim.iloc[idx]["onset"]
            
            # Rest interval: [event_onset + 15s, next_event_onset - 5s]
            rest_start = current_onset + 15.0
            
            if idx + 1 < len(rec_stim):
                next_onset = rec_stim.iloc[idx + 1]["onset"]
                rest_end = next_onset - 5.0
            else:
                rest_end = recording_end
            
            interval_length = rest_end - rest_start
            
            # Check if interval is long enough for at least one segment
            if interval_length < segment_length_sec:
                print(f"  Skipping rest interval {idx}: [{rest_start:.1f}s - {rest_end:.1f}s] (length: {interval_length:.1f}s < {segment_length_sec}s)")
                continue
            
            print(f"  Rest interval {idx}: [{rest_start:.1f}s - {rest_end:.1f}s] (length: {interval_length:.1f}s)")
            
            # First pass: collect valid rest segment starts in this interval
            t = rest_start
            valid_segment_starts = []
            
            while t + segment_length_sec <= rest_end:
                baseline_start = max(t - baseline_duration, recording_start)
                baseline_rest = delta_brain.sel(time=slice(baseline_start, t)).mean("time")
                x_rest = delta_brain.sel(time=slice(t, t + segment_length_sec)) - baseline_rest
                x_rest = x_rest.isel(time=slice(0, n_timepoints))
                
                # Skip if segment is too short
                if x_rest.sizes["time"] < n_timepoints:
                    print(f"    Skipping rest segment at t={t:.1f}s (insufficient samples: {x_rest.sizes['time']} < {n_timepoints})")
                    t += step_sec
                    continue
                
                valid_segment_starts.append(t)
                t += step_sec
            
            if len(valid_segment_starts) == 0:
                continue
            
            # Select the segment centered closest to interval midpoint as test
            interval_midpoint = (rest_start + rest_end) / 2.0
            segment_centers = np.array(valid_segment_starts) + (segment_length_sec / 2.0)
            test_segment_idx = int(np.argmin(np.abs(segment_centers - interval_midpoint)))
            
            interval_segments = 0
            for seg_idx, t_start in enumerate(valid_segment_starts):
                baseline_start = max(t_start - baseline_duration, recording_start)
                baseline_rest = delta_brain.sel(time=slice(baseline_start, t_start)).mean("time")
                x_rest = delta_brain.sel(time=slice(t_start, t_start + segment_length_sec)) - baseline_rest
                x_rest = x_rest.isel(time=slice(0, n_timepoints))
                x_rest = x_rest.transpose("parcel", "chromo", "time")
                del x_rest.time.attrs['units']
                
                is_test_segment = seg_idx == test_segment_idx
                rest_suffix = "_test.nc" if is_test_segment else ".nc"
                rest_filename = file.replace(str(pre_processed_path), str(processed_path)).replace(".pkl", f"_{rest_label}_{rest_segment_count}{rest_suffix}")
                x_rest.to_netcdf(rest_filename)
                rest_segment_count += 1
                interval_segments += 1
            
            print(f"  Extracted {interval_segments} rest segments from interval {idx} (middle segment index {test_segment_idx} saved as _test.nc)")
    
        print(f"Total rest segments extracted: {rest_segment_count}")

    print("finished processing file: ", os.path.basename(file).replace(".pkl",".npy"))

print('\n--- Done!---')

In [ ]:
file_to_load = "datasets/processed/BS_Laura/motor_90chs/sub-538/sub-538_task-BS_run-01_nirs_left_3.nc"
ds = xr.load_dataset(file_to_load)
ds

In [ ]:
from collections import defaultdict
import pandas as pd
segmented_files = glob.glob(str(processed_path / 'sub-*' / '*.nc'))

subject_label_counts = defaultdict(lambda: defaultdict(int))

for file_path in segmented_files:
    filename = os.path.basename(file_path)
    subject = os.path.basename(os.path.dirname(file_path))
    
    if '_task_' in filename:
        label = 'task'
    elif '_rest_' in filename:
        label = 'rest'
    else:
        label = 'unknown'
    
    subject_label_counts[subject][label] += 1

data_for_df = []
for subject in sorted(subject_label_counts.keys()):
    counts = subject_label_counts[subject]
    data_for_df.append({
        'Subject': subject,
        'Task': counts.get('task', 0),
        'Rest': counts.get('rest', 0),
        'Total': counts.get('task', 0) + counts.get('rest', 0)
    })

df_counts = pd.DataFrame(data_for_df)

print(df_counts.to_string(index=False))

print("="*40)
print(f"Total subjects: {len(df_counts)}")
print(f"Total task segments: {df_counts['Task'].sum()}")
print(f"Total rest segments: {df_counts['Rest'].sum()}")
print(f"Total segments: {df_counts['Total'].sum()}")
print(f"\nClass balance: Task={df_counts['Task'].sum()}, Rest={df_counts['Rest'].sum()}")
print(f"Task/Rest ratio: {df_counts['Task'].sum() / df_counts['Rest'].sum():.2f}" if df_counts['Rest'].sum() > 0 else "Task/Rest ratio: N/A")

In [ ]:
test_files = sorted(processed_path.rglob("*_test.nc"))

counts = {"task": 0, "rest": 0}
for f in test_files:
    name = f.name.lower()
    if "_rest_" in name:
        counts["rest"] += 1
    else:
        counts["task"] += 1

df_test_counts = pd.DataFrame([
    {"Label": "task", "_test_files": counts["task"]},
    {"Label": "rest", "_test_files": counts["rest"]},
])

display(df_test_counts)
print(f"Total _test files: {len(test_files)}")


### channel space

In [ ]:
# deprecated code for channel space processing. We are now doing image reconstruction and then selecting parcels based on sensitivity, thus we don't need to do this channel space processing anymore. However, I keep this code here for reference and in case we want to compare with channel space results in the future.

baseline_duration = 2.5  # in seconds
n_shifts = 9
duration = 10  # in seconds
post_padding = 5  # in seconds
n_timepoints = 87  # fixed length after shifting

if DATASET_NAME == "BallSqueezingHD_modified":
    delta_range = (-2.5, 2.5)
elif DATASET_NAME == "FreshMotor":
    delta_range = (-2.0, 0.0)
start_shift = np.linspace(*delta_range, n_shifts)

    
for file in proc_pkl_files:
    with open(file, 'rb') as handle:
        data_pickle = pickle.load(handle)
    
    
    rec_stim = data_pickle['rec_stim']
    conc_pcr = data_pickle['conc_pcr'] # channel space data

    # Align subject-specific parcels to a common parcel template (zero-pad missing parcels)
    # delta_brain = delta_brain.sel(parcel=sensitive_parcels).reindex(parcel=PARCEL_TEMPLATE, fill_value=0)
    
    i = 0
    for index, row in rec_stim.iterrows():
        label = row["trial_type"].lower()
        for s in start_shift:
            start_time = row["onset"] + s
            end_time = start_time + duration + post_padding # in seconds
            
            # channel space baseline
            baseline_conc_pcr = conc_pcr.sel(
                time=slice(row["onset"] - baseline_duration, row["onset"])
            ).mean("time")
        
            # channel space segment
            x_channel = conc_pcr.sel(time=slice(start_time, end_time)) - baseline_conc_pcr
            x_channel = x_channel.isel(time=slice(0, n_timepoints))
            x_channel = x_channel.transpose("channel", "chromo", "time")

            # print(x_channel.shape)
            del x_channel.time.attrs['units']
            if not os.path.exists(os.path.dirname(file.replace(str(pre_processed_path), str(processed_path)))):
                os.makedirs(os.path.dirname(file.replace(str(pre_processed_path), str(processed_path))))
            if s == 0:
                x_channel.to_netcdf(file.replace(str(pre_processed_path), str(processed_path)).replace(".pkl", "_" + label + "_"+str(i)+"_test.nc"))
                i += 1
            else:
                x_channel.to_netcdf(file.replace(str(pre_processed_path), str(processed_path)).replace(".pkl", "_" + label + "_"+str(i)+".nc"))
                i += 1

    print("finished processing file: ", os.path.basename(file).replace(".pkl",".npy"))
print('--- Done!---')            

In [ ]:
# load a specific .nc file
file_to_load = "datasets/processed/Anderson_sparse/sub-1/sub-1_ses-1_task-WordStroop_run-1_nirs_wordcongruent_0.nc"
ds = xr.load_dataset(file_to_load)
ds

file_to_load = "datasets/processed/vfc_hd/sub-01/sub-01_ses-02_task-WordStroop_run-01_nirs_wordcongruent_0.nc"
ds = xr.load_dataset(file_to_load)
ds



### Old appraoch using event files and freq0.5

FreshMotor

- parcel space: (parcel: 110, chromo: 2, time: 87)
- channel space: (time: 87, channel: 68, chromo: 2)

---
BallSqueezing

- parcel space: (parcel: 110, chromo: 2, time: 87)
- channel space: (channel: 100, chromo: 2, time: 87)


In [ ]:
# FRESHMOTOR dataset
# load one processed nc file
filename = 'datasets/processed/FreshMotor/sub-01/sub-01_task-FRESHMOTOR_run-left2s_nirs_left_0.nc'
data = xr.load_dataset(filename)
    
# extract np array from xarray and remove first dimension
# data.to_array().squeeze().transpose("channel", "chromo", "time")
data

In [ ]:
type(data)
getattr(data, "dims", None)
[data[v].dims for v in data.data_vars]  # if Dataset



# Outdated approach

In [ ]:
baseline_duration = 2.5  # in seconds
n_shifts = 9
duration = 10  # in seconds
post_padding = 5  # in seconds
n_timepoints = 87  # fixed length after shifting

if DATASET_NAME == "BallSqueezingHD_modified":
    delta_range = (-2.5, 2.5)
elif DATASET_NAME == "FreshMotor":
    delta_range = (-2.0, 0.0)
start_shift = np.linspace(*delta_range, n_shifts)






label_dict = {'right':1, 'left':2}  
subject_to_rec = {}            
INDEX = 0     
freq_dir = processed_path / f'frq{0.5}'
freq_dir.mkdir(exist_ok=True)       
for file in proc_pkl_files:
    with open(file, 'rb') as handle:
        data_pickle = pickle.load(handle)
    
    delta_brain = data_pickle['delta_conc']
    sensitive_parcels = data_pickle['sensitive_parcels']
    rec_stim = data_pickle['rec_stim']

    # Align subject-specific parcels to a common parcel template (zero-pad missing parcels)
    # delta_brain = delta_brain.sel(parcel=sensitive_parcels).reindex(parcel=PARCEL_TEMPLATE, fill_value=0)
    delta_brain = delta_brain.sel(parcel=template_sens_parcel_list)

    i = 0
    
    
    
    
    # ------
    SUB = PureWindowsPath(file).parts[-2]
    if SUB not in subject_to_rec:
        subject_to_rec[SUB] = []
    try:
        sub_dir = Path(freq_dir) / SUB
        sub_dir.mkdir(parents=True, exist_ok=True)
    except FileExistsError:
        pass
    # ------

    for index, row in rec_stim.iterrows():
        label = row["trial_type"].lower()
        for s in start_shift:
            start_time = row["onset"] + s
            end_time = start_time + duration + post_padding # in seconds
            baseline = delta_brain.sel(
                time=slice(row["onset"] - baseline_duration, row["onset"])
            ).mean("time")
            
            # Then, trimming is easy with `.sel()`:
            x = delta_brain.sel(time=slice(start_time, end_time)) - baseline
            x = x.isel(time=slice(0, n_timepoints))
            x = x.transpose("parcel", "chromo", "time")
            del x.time.attrs['units']

            data = {
                'xt': x,
                'file': file,
                'class': label_dict[label],
            }

            # -----
            # save events
            filename = r'{}/{}/event_{}_delta{:+.2f}_{}.pkl'.format(freq_dir, SUB, 'aug' if s != 0.0 else 'orig', s, INDEX)
            subject_to_rec[SUB].append(filename)
            INDEX += 1
            # -----
            
            
            # if not os.path.exists(os.path.dirname(file.replace(str(pre_processed_path), str(processed_path)))):
            #     os.makedirs(os.path.dirname(file.replace(str(pre_processed_path), str(processed_path))))
            # if s == 0:
            #     x.to_netcdf(file.replace(str(pre_processed_path), str(processed_path)).replace(".pkl", "_" + label + "_"+str(i)+"_test.nc"))
            #     i += 1
            # else:
            #     x.to_netcdf(file.replace(str(pre_processed_path), str(processed_path)).replace(".pkl", "_" + label + "_"+str(i)+".nc"))
            #     i += 1
            with open(filename, 'wb') as handle:
                pickle.dump(data, handle, protocol=pickle.HIGHEST_PROTOCOL)
                
with open('{}/meta_event_{}.pkl'.format(freq_dir, 0.5), 'wb') as handle:
    pickle.dump(subject_to_rec, handle, protocol=pickle.HIGHEST_PROTOCOL)
print('--- Done!---')            
    # print("finished processing file: ", os.path.basename(file).replace(".pkl",".npy"))

In [ ]:
events = "datasets/processed/{}/frq{}/meta_event_{}.pkl"
    
meta_events = []

with open(events.format(DATASET_NAME, 0.5, 0.5), 'rb') as handle:
    meta = pickle.load(handle)
meta_events.append(meta)


if DATASET_NAME == "BallSqueezingHD_modified":
    session_to_files = {'run-1':[],
                        'run-2':[],
                        'run-3':[]}
    
elif DATASET_NAME == "FreshMotor":    
    session_to_files = {'run-left2s':[],
                        'run-right2s':[],
                        'run-left3s':[],
                        'run-right3s':[]}

files_to_session = {}
for meta_event in meta_events:
    for sub in meta_event:
        for file in meta_event[sub]:
            meta = None
            with open(file, 'rb') as handle:
                meta = pickle.load(handle) 
            
            run = meta['file'].split('_')[-2]
            files_to_session[file] = run
            session_to_files[run].append(file)

for run in session_to_files:
    print(run, len(session_to_files[run]))

# this will save the mapping of files to sessions used for LOSO
with open(f'datasets/processed/{DATASET_NAME}/files_to_sessions.pkl', 'wb') as handle:
    pickle.dump(files_to_session, handle, protocol=pickle.HIGHEST_PROTOCOL)     
print("Saved files_to_sessions.pkl")